In [4]:
import os

In [5]:
!pip install -q youtube-transcript-api langchain-community langchain-openai\
            faiss-cpu tiktoken python-dotenv

In [6]:
import langchain
print(langchain.__version__)

1.3.11


In [7]:
pip install -U langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [9]:
!pip install langchain
!pip install -U langchain langchain-core langchain-community langchain-openai faiss-cpu
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

C:\Users\hp\AppData\Local\Temp\ipykernel_11464\199892103.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [12]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

video_url = "https://www.youtube.com/watch?v=PDoYP4LqDdY&list=PL2RWXl0vclRJ0WTFafqTMTBovzwr6bqeT"
if "v=" in video_url:
    video_id = video_url.split("v=")[1].split("&")[0]
else:
    video_id = video_url.split("/")[-1]

try:
    api_instance = YouTubeTranscriptApi()
    
    # Use the .fetch() method on the instance
    transcript_data = api_instance.fetch(video_id, languages=["en"])
    transcript = " ".join(snippet.text for snippet in transcript_data.snippets)
    
    print(transcript[:500]) 
    
except TranscriptsDisabled:
    print("No captions available for this video.")

aunts there lived a poor farmer [Music] he had a goose it laid one golden egg every day the former sold the eggs and became rich [Music] he was a greedy farmer he wanted all the golden eggs at the same time [Music] so he took a knife and cut the goose foreign but found only one egg inside it as the greedy farmer lost both the goose and golden eggs and became poor again


In [13]:
splitter=RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10)
chunks= splitter.create_documents([transcript])

In [14]:
%pip install langchain-huggingface sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [15]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vector_store= FAISS.from_documents(chunks,embeddings)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [16]:
vector_store.index_to_docstore_id

{0: '0704510a-86e9-440f-9ce0-c2d8518b603f',
 1: '7b6eab35-f9ef-4f13-a6d7-af94806a82af',
 2: '0a845429-e527-402b-b8f8-7657958f7030',
 3: 'a616a441-32cc-4180-be40-87f247b32184',
 4: '7e00f21a-b480-4f01-bb99-20fe9ac96fcc',
 5: '8831e449-c0ab-4b5e-ba05-982b96778c33',
 6: 'a068b6c9-025b-4576-b3e1-d2e0863b00df',
 7: 'c99f92dd-11f4-438b-9b6f-a348f718b08b',
 8: 'c576b89c-afad-424c-a0e3-2620ecd7db0b',
 9: '4675f457-1297-4e29-bc87-103e74a2718b'}

In [17]:
print(vector_store.docstore._dict)

{'0704510a-86e9-440f-9ce0-c2d8518b603f': Document(id='0704510a-86e9-440f-9ce0-c2d8518b603f', metadata={}, page_content='aunts there lived a poor farmer [Music] he had a'), '7b6eab35-f9ef-4f13-a6d7-af94806a82af': Document(id='7b6eab35-f9ef-4f13-a6d7-af94806a82af', metadata={}, page_content='he had a goose it laid one golden egg every day'), '0a845429-e527-402b-b8f8-7657958f7030': Document(id='0a845429-e527-402b-b8f8-7657958f7030', metadata={}, page_content='every day the former sold the eggs and became'), 'a616a441-32cc-4180-be40-87f247b32184': Document(id='a616a441-32cc-4180-be40-87f247b32184', metadata={}, page_content='became rich [Music] he was a greedy farmer he'), '7e00f21a-b480-4f01-bb99-20fe9ac96fcc': Document(id='7e00f21a-b480-4f01-bb99-20fe9ac96fcc', metadata={}, page_content='farmer he wanted all the golden eggs at the same'), '8831e449-c0ab-4b5e-ba05-982b96778c33': Document(id='8831e449-c0ab-4b5e-ba05-982b96778c33', metadata={}, page_content='the same time [Music] so he took

In [18]:
retriever=vector_store.as_retriever(search_type="similarity",search_kwargs={"k":2})

In [19]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000255325FDE80>, search_kwargs={'k': 2})

In [20]:
retriever.invoke('What is the animal in the description')

[Document(id='7b6eab35-f9ef-4f13-a6d7-af94806a82af', metadata={}, page_content='he had a goose it laid one golden egg every day'),
 Document(id='c576b89c-afad-424c-a0e3-2620ecd7db0b', metadata={}, page_content='lost both the goose and golden eggs and became')]

In [21]:
!pip install langchain langchain-ollama

In [22]:
from langchain_ollama import ChatOllama
llm=ChatOllama(model="llama3.2",temperature=0.2)

In [23]:
prompt=PromptTemplate(
    template="""
       You are a helpful assistant.
       Answer ONLY from the provided transcript context.
       If the context is insufficient answer you don't know

         {context}
        Question:{question}
    """ ,
     input_variables = ['context','question']
)

In [50]:
question= "According to the provided transcript, which animal did the boy have? "
retrieve_docs= retriever.invoke(question)

In [51]:
retrieve_docs

[Document(id='7b6eab35-f9ef-4f13-a6d7-af94806a82af', metadata={}, page_content='he had a goose it laid one golden egg every day'),
 Document(id='c576b89c-afad-424c-a0e3-2620ecd7db0b', metadata={}, page_content='lost both the goose and golden eggs and became')]

In [52]:
context_text="\n\n".join(doc.page_content for doc in retrieve_docs)

In [53]:
context_text

'he had a goose it laid one golden egg every day\n\nlost both the goose and golden eggs and became'

In [54]:
final_prompt=prompt.invoke({"context":context_text,"question":question})

In [55]:
final_prompt

StringPromptValue(text="\n       You are a helpful assistant.\n       Answer ONLY from the provided transcript context.\n       If the context is insufficient answer you don't know\n\n         he had a goose it laid one golden egg every day\n\nlost both the goose and golden eggs and became\n        Question:According to the provided transcript, which animal did the boy have? \n    ")

In [56]:
answer=llm.invoke(final_prompt)
print(answer.content)

A goose.


In [60]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [63]:
def format_docs(retrieve_docs):
    context_text="\n\n".join(doc.page_content for doc in retrieve_docs),
    return context_text

In [66]:
parallel_chain=RunnableParallel({
    'context':retriever|RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [69]:
parallel_chain.invoke("What animal is depicted in the story?")

{'context': ('he had a goose it laid one golden egg every day\n\nfarmer he wanted all the golden eggs at the same',),
 'question': 'What animal is depicted in the story?'}

In [70]:
parser=StrOutputParser()

In [71]:
main_chain=parallel_chain|prompt|llm|parser

In [73]:
main_chain.invoke("Mention the animal in the context")

'The animal mentioned in the context is a goose.'